# Datasets and DataLoaders

`torch.utils.data` gives us:

- **Dataset** — one sample at a time (`__len__`, `__getitem__`)
- **DataLoader** — batches, shuffle, optional worker processes

That combination is how we feed a training loop without loading the whole dataset into one giant tensor of collated batches by hand.


In [ ]:
import os
from pathlib import Path

import numpy as np
import torch
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from torch.nn.utils.rnn import pad_sequence
import torchvision.transforms as transforms

torch.manual_seed(0)


## 1. A minimal custom Dataset

`__getitem__` must return tensors (or something the default collate function can stack).


In [ ]:
class SimpleCustomDataset(Dataset):
    def __init__(self, features, labels):
        assert len(features) == len(labels)
        self.features = features
        self.labels = labels

    def __len__(self):
        return len(self.features)

    def __getitem__(self, idx):
        feature = torch.as_tensor(self.features[idx], dtype=torch.float32)
        label = torch.as_tensor(self.labels[idx], dtype=torch.long)
        return feature, label


features = np.random.randn(20, 4).astype(np.float32)
labels = np.random.randint(0, 2, size=20)
dataset = SimpleCustomDataset(features, labels)
x0, y0 = dataset[0]
print(len(dataset), x0.shape, y0)


## 2. DataLoader: batching and shuffle

- Training: `shuffle=True` so batches are not in a fixed order.
- Validation / test: `shuffle=False` so metrics are comparable across runs.
- `drop_last=True` discards an incomplete last batch.
- `num_workers>0` loads data in background processes (a common starting point on GPU is `4 * num_gpus`).
- `pin_memory=True` speeds up CPU → GPU copies. It does nothing useful on CPU-only training.


In [ ]:
class DummyDataset(Dataset):
    def __init__(self, num_samples=105):
        self.features = torch.randn(num_samples, 10)
        self.labels = torch.randint(0, 2, (num_samples,))

    def __len__(self):
        return self.features.size(0)

    def __getitem__(self, idx):
        return self.features[idx], self.labels[idx]


dataset = DummyDataset()
train_loader = DataLoader(dataset, batch_size=32, shuffle=True)

for epoch in range(1):
    print(f"epoch {epoch + 1}")
    for i, (batch_x, batch_y) in enumerate(train_loader):
        print(f"  batch {i + 1}: features {tuple(batch_x.shape)}, labels {tuple(batch_y.shape)}")

# GPU-oriented loader (safe to construct on CPU; pin_memory only helps with CUDA)
_fast_loader = DataLoader(
    dataset, batch_size=32, shuffle=True, num_workers=0, pin_memory=torch.cuda.is_available()
)


## 3. Transforms on synthetic features

`torchvision.transforms.Compose` can wrap any callable. Here we normalize tabular features.


In [ ]:
num_samples, num_features = 100, 10
features = torch.randn(num_samples, num_features)
labels = torch.randint(0, 2, (num_samples,))

feature_mean = features.mean(dim=0)
feature_std = features.std(dim=0)
feature_std[feature_std == 0] = 1.0


class ToTensorAndType:
    def __call__(self, sample):
        feature, label = sample["feature"], sample["label"]
        return {"feature": feature.float(), "label": label.long()}


class NormalizeFeatures:
    def __init__(self, mean, std):
        self.mean = mean
        self.std = std

    def __call__(self, sample):
        feature, label = sample["feature"], sample["label"]
        return {"feature": (feature - self.mean) / self.std, "label": label}


class SyntheticDataset(Dataset):
    def __init__(self, features, labels, transform=None):
        assert features.shape[0] == labels.shape[0]
        self.features = features
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return self.features.shape[0]

    def __getitem__(self, idx):
        sample = {"feature": self.features[idx], "label": self.labels[idx]}
        if self.transform:
            sample = self.transform(sample)
        return sample["feature"], sample["label"]


data_transforms = transforms.Compose(
    [ToTensorAndType(), NormalizeFeatures(mean=feature_mean, std=feature_std)]
)
transformed_dataset = SyntheticDataset(features, labels, transform=data_transforms)

sample_x, sample_y = transformed_dataset[0]
print("original:", features[0][:4])
print("normalized:", sample_x[:4])

loader = DataLoader(transformed_dataset, batch_size=16, shuffle=True, num_workers=0)
feature_batch, label_batch = next(iter(loader))
print("batch:", feature_batch.shape, label_batch.shape, feature_batch.dtype, label_batch.dtype)


## 4. Image-style transforms (reference)

These pipelines are the usual ImageNet recipe. They need PIL images; we only **print** them here so the notebook stays offline-friendly.

```python
train_transforms = transforms.Compose([
    transforms.Resize(256),
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])
```

Built-in datasets (`CIFAR10`, `ImageFolder`) work the same way:

```python
from torchvision.datasets import CIFAR10, ImageFolder

train_dataset = CIFAR10(root="./data", train=True, download=True, transform=transforms.ToTensor())
folder = ImageFolder(root=Path("path/to/train_images"), transform=train_transforms)
```

For audio use `torchaudio`; for text, Hugging Face `datasets` or `torchtext`.


## 5. Custom image Dataset (pattern)

Pattern for a list of file paths. This cell does not open files so it always runs.


In [ ]:
from PIL import Image


class CustomImageDataset(Dataset):
    def __init__(self, image_paths, labels, transform=None):
        self.image_paths = image_paths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        image = Image.open(self.image_paths[idx]).convert("RGB")
        if self.transform:
            image = self.transform(image)
        elif not isinstance(image, torch.Tensor):
            image = torch.as_tensor(np.array(image), dtype=torch.float32).permute(2, 0, 1) / 255.0
        label = torch.as_tensor(self.labels[idx], dtype=torch.long)
        return image, label


print("CustomImageDataset defined. We pass real paths when we have images on disk.")


## 6. `WeightedRandomSampler` (imbalanced classes)

Each sample gets weight `1 / class_count`. `replacement=True` draws with replacement so minority classes appear more often.


In [ ]:
targets = [0, 1, 0, 0, 0, 1, 1, 0, 1, 1, 1, 0, 0, 1]
class_counts = torch.bincount(torch.tensor(targets))
sample_weights = torch.tensor([1.0 / class_counts[t] for t in targets], dtype=torch.double)
sampler = WeightedRandomSampler(weights=sample_weights, num_samples=len(targets), replacement=True)

# When we use a sampler, we do not also set shuffle=True
balanced_loader = DataLoader(dataset, batch_size=8, sampler=sampler)
print(next(iter(balanced_loader))[1])


## 7. `collate_fn` for variable-length sequences

The default collate stacks equal-shaped tensors. Sequences of different lengths need padding.


In [ ]:
class VariableSequenceDataset(Dataset):
    def __init__(self, data):
        self.data = data

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        sequence = self.data[idx]
        label = len(sequence)
        return sequence, label


def pad_collate(batch):
    sequences = [item[0] for item in batch]
    labels = torch.tensor([item[1] for item in batch])
    padded = pad_sequence(sequences, batch_first=True, padding_value=0.0)
    return padded, labels


sequences = [torch.randn(int(torch.randint(5, 15, (1,)).item())) for _ in range(20)]
seq_dataset = VariableSequenceDataset(sequences)
seq_loader = DataLoader(seq_dataset, batch_size=4, collate_fn=pad_collate)
padded_batch, length_batch = next(iter(seq_loader))
print("padded batch:", padded_batch.shape, "lengths:", length_batch)
